# Hypothesis Testing: Caral‑Supe Formation and Decline
Using the SUBIT heuristic framework and the fact database.

In [1]:
import sys, os, sqlite3, pandas as pd

# Absolute path to the database – adjust if your project folder is different
DB_PATH = r'C:\Users\sciga\subit-civ\caral_facts.sqlite'
if not os.path.exists(DB_PATH):
    # Fallback: try relative path (notebook run from inside notebooks/)
    DB_PATH = '../caral_facts.sqlite'

print('Using database:', os.path.abspath(DB_PATH))
conn = sqlite3.connect(DB_PATH)
print('Connected.')

Using database: C:\Users\sciga\subit-civ\caral_facts.sqlite
Connected.


In [2]:
# Load experiments
exp_df = pd.read_sql("SELECT * FROM experiments", conn)
exp_df

,experiment_uuid,title,research_question,null_hypothesis_rule_id,alternative_hypothesis_rule_id
0,faeeeb8e-46f4-4459-b2c4-f93abd1c0b77,Testing Formative Phase Hypotheses,Which subsystem drove monumentality first?,7,7
1,4c75cfb3-678f-491b-82b8-45f5e569c815,Testing Decline Phase Hypotheses,Sudden collapse or managed adaptation?,8,10


In [3]:
# For each experiment, show discriminating tests
for _, row in exp_df.iterrows():
    exp_uuid = row['experiment_uuid']
    title = row['title']
    tests = pd.read_sql(f"SELECT * FROM discriminating_tests WHERE experiment_uuid = '{exp_uuid}'", conn)
    print(f"Experiment: {title}")
    display(tests)
    print()

Experiment: Testing Formative Phase Hypotheses


,test_uuid,experiment_uuid,description,expected_result_if_null,expected_result_if_alternative
0,test-form-001,faeeeb8e-46f4-4459-b2c4-f93abd1c0b77,Isotopes: marine protein % at Caral vs Aspero,similar (>40%),Caral < 30%
1,test-form-002,faeeeb8e-46f4-4459-b2c4-f93abd1c0b77,Site size: inland vs coastal,similar sizes,inland much larger
2,test-form-003,faeeeb8e-46f4-4459-b2c4-f93abd1c0b77,Earliest radiocarbon dates,coastal older or same,inland older



Experiment: Testing Decline Phase Hypotheses


,test_uuid,experiment_uuid,description,expected_result_if_null,expected_result_if_alternative
0,test-decl-001,4c75cfb3-678f-491b-82b8-45f5e569c815,"Destruction layer (thin, synchronous)","present, same time everywhere",absent or asynchronous
1,test-decl-002,4c75cfb3-678f-491b-82b8-45f5e569c815,Abandonment timing (inland vs coastal),inland first,coastal first or simultaneous
2,test-decl-003,4c75cfb3-678f-491b-82b8-45f5e569c815,Cultural continuity at Vichama/Peñico,abrupt change,continuity in architecture/symbols


In [53]:
# Hypothesis evaluation: test-form-001 (marine protein %)
# Load marine protein observations
conn_check = sqlite3.connect(DB_PATH)   # fresh connection to be safe
obs_marine = pd.read_sql("SELECT * FROM observations WHERE type='marine_protein_%'", conn_check)
print(f"Found {len(obs_marine)} marine protein observations.")
display(obs_marine)
conn_check.close()

Found 2 marine protein observations.


,obs_id,site_id,source_id,type,value,year_from,year_to,method
0,18,1,5,marine_protein_%,25.0,-2700,-2000,Pezo-Lanfranco2022
1,19,2,5,marine_protein_%,45.0,-2700,-2000,Pezo-Lanfranco2022


In [54]:
# Evaluate the test
conn_eval = sqlite3.connect(DB_PATH)
obs_marine_eval = pd.read_sql("SELECT * FROM observations WHERE type='marine_protein_%'", conn_eval)
caral_marine = obs_marine_eval[obs_marine_eval['site_id'] == 1]['value']
aspero_marine = obs_marine_eval[obs_marine_eval['site_id'] == 2]['value']

if not caral_marine.empty:
    mean_caral = caral_marine.mean()
    print(f"Mean marine protein at Caral: {mean_caral:.1f}%")
    if mean_caral < 30:
        print("Result: Supports alternative hypothesis (agricultural primacy)")
    else:
        print("Result: Supports null hypothesis (complementarity)")
else:
    print("No marine protein data for Caral.")

if not aspero_marine.empty:
    print(f"Mean marine protein at Aspero: {aspero_marine.mean():.1f}%")
conn_eval.close()

Mean marine protein at Caral: 25.0%
Result: Supports alternative hypothesis (agricultural primacy)
Mean marine protein at Aspero: 45.0%


In [55]:
# Additional test: test-decl-001 (destruction layer) — using abandonment dates as proxy
conn_abandon = sqlite3.connect(DB_PATH)
obs_abandon = pd.read_sql("SELECT * FROM observations WHERE type IN ('abandonment_year','founding_year')", conn_abandon)
print(f"Found {len(obs_abandon)} abandonment/founding records.")
display(obs_abandon)
conn_abandon.close()

Found 3 abandonment/founding records.


,obs_id,site_id,source_id,type,value,year_from,year_to,method
0,22,1,4,abandonment_year,-1850.0,-1800,Shady2025_Caral,None
1,23,2,3,abandonment_year,-1900.0,-1850,Sandweiss2009_Aspero,None
2,24,3,4,founding_year,-1750.0,-1700,Shady2025_Vichama,None


In [56]:
# Simple logic: if all abandonment dates are within a narrow window, supports seismic shock (synchronous)
conn_sync = sqlite3.connect(DB_PATH)
obs_abandon_sync = pd.read_sql("SELECT * FROM observations WHERE type IN ('abandonment_year','founding_year')", conn_sync)
abandon_dates = obs_abandon_sync[obs_abandon_sync['type'] == 'abandonment_year']
if len(abandon_dates) >= 2:
    range_years = abandon_dates['year_from'].max() - abandon_dates['year_from'].min()
    print(f"Abandonment date range: {range_years} years")
    if range_years <= 100:
        print("Interpretation: relatively synchronous → consistent with seismic shock")
    else:
        print("Interpretation: spread over >100 years → more consistent with gradual processes")
else:
    print("Not enough abandonment data to test.")
conn_sync.close()

Abandonment date range: 50 years
Interpretation: relatively synchronous → consistent with seismic shock


In [57]:
# Close any remaining connection (if needed)
try:
    conn.close()
except:
    pass

In [58]:
# Check radiocarbon dates
conn_radio = sqlite3.connect(DB_PATH)
radio = pd.read_sql("SELECT * FROM observations WHERE type='earliest_radiocarbon_date'", conn_radio)
display(radio)
conn_radio.close()

,obs_id,site_id,source_id,type,value,year_from,year_to,method
0,27,1,6,earliest_radiocarbon_date,-2627.0,-2700,-2550,Shady2001_Caral
1,28,2,6,earliest_radiocarbon_date,-2550.0,-2700,-2400,Shady2001_Aspero


In [59]:
# Compare inland vs coastal abandonment
conn_ab2 = sqlite3.connect(DB_PATH)
ab2 = pd.read_sql("SELECT sites.name, observations.* FROM observations JOIN sites ON observations.site_id = sites.site_id WHERE observations.type='abandonment_year'", conn_ab2)
display(ab2)
conn_ab2.close()

,name,obs_id,site_id,source_id,type,value,year_from,year_to,method
0,Caral,22,1,4,abandonment_year,-1850.0,-1800,Shady2025_Caral,NaN
1,Aspero,23,2,3,abandonment_year,-1900.0,-1850,Sandweiss2009_Aspero,NaN
2,Huaricanga,25,5,6,abandonment_year,-2000.0,-2100,-1900,Haas2004_Huaricanga
3,Bandurria,26,6,3,abandonment_year,-1900.0,-2000,-1800,Sandweiss2009_Bandurria


In [4]:
# Test cultural continuity
conn_cult = sqlite3.connect(DB_PATH)
cult = pd.read_sql("SELECT * FROM observations WHERE type='cultural_continuity_index'", conn_cult)
display(cult)
if len(cult) > 0:
    if cult['value'].mean() > 0.5:
        print("Cultural continuity present → supports alternative hypothesis (managed migration)")
    else:
        print("Cultural break → supports null hypothesis (seismic shock)")
conn_cult.close()

,obs_id,site_id,source_id,type,value,year_from,year_to,method
0,29,3,4,cultural_continuity_index,1.0,-1750,-1700,Shady2025_Vichama_architecture
1,30,4,4,cultural_continuity_index,1.0,-1750,-1700,Shady2025_Penico_rituals


Cultural continuity present → supports alternative hypothesis (managed migration)
